# 🔍 Module de Recherche Sémantique RAG — PostgreSQL + pgvector
### STE AGRO MELANGE TECHNOLOGIE — ROSE BLANCHE Group  
### Assistance intelligente à la formulation en boulangerie / pâtisserie

**Architecture :**
- Base : PostgreSQL — table `embeddings (id, id_document, texte_fragment, vecteur VECTOR(384))`
- Modèle imposé : `all-MiniLM-L6-v2`  (sentence-transformers, dim=**384**)
- Méthode de similarité : **Cosine Similarity** via pgvector
- Résultats : **Top K = 3**

$$\text{cosine\_similarity}(Q, D) = 1 - \frac{Q \cdot D}{\|Q\| \cdot \|D\|}$$


## 1. Installation et importation des bibliothèques

In [ ]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install sentence-transformers psycopg2-binary numpy pandas

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import psycopg2
import psycopg2.extras
from sentence_transformers import SentenceTransformer

# ── Constantes imposées par le cahier des charges ───────────────────────────
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # modèle officiel
EMBEDDING_DIM   = 384                   # dimension du vecteur
TOP_K           = 3                     # nombre de résultats à retourner

print("✅ Bibliothèques importées avec succès.")
print(f"   Modèle imposé  : {EMBEDDING_MODEL}")
print(f"   Dimension      : {EMBEDDING_DIM}")
print(f"   Top-K          : {TOP_K}")


## 2. Connexion à la base PostgreSQL

La base contient déjà les embeddings des fiches techniques dans la table `embeddings` :

| Colonne          | Type           | Description                         |
|------------------|----------------|-------------------------------------|
| `id`             | Primary Key    | Identifiant unique du fragment       |
| `id_document`    | int            | Référence au document source         |
| `texte_fragment` | text           | Texte du fragment (chunk)            |
| `vecteur`        | VECTOR(384)    | Embedding pré-calculé                |

> ✏️ **Renseignez vos paramètres de connexion dans la cellule ci-dessous.**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ✏️  MODIFIEZ ICI VOS PARAMÈTRES DE CONNEXION
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "boulangerie_db"
DB_USER     = "postgres"
DB_PASSWORD = ""             # ← votre mot de passe
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

try:
    conn = psycopg2.connect(
        host=DB_HOST, port=DB_PORT,
        dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
    )
    print(f"✅ Connexion réussie → {DB_NAME} sur {DB_HOST}:{DB_PORT}")

    # Vérification : nombre de fragments en base
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM embeddings;")
        n_total = cur.fetchone()[0]
        cur.execute("SELECT COUNT(DISTINCT id_document) FROM embeddings;")
        n_docs = cur.fetchone()[0]

    print(f"   Fragments indexés  : {n_total:,}")
    print(f"   Documents uniques  : {n_docs:,}")

    # Aperçu des premières lignes
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute("""
            SELECT id, id_document,
                   LEFT(texte_fragment, 100) AS apercu
            FROM embeddings
            LIMIT 3;
        """)
        rows = cur.fetchall()

    print("\n─── Aperçu des 3 premiers fragments ───")
    for row in rows:
        print(f"  id={row['id']}  doc={row['id_document']}  | {row['apercu']}…")

except psycopg2.OperationalError as e:
    print(f"❌ Connexion impossible : {e}")
    print("   Vérifiez DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD.")


## 3. Chargement du modèle d'embedding `all-MiniLM-L6-v2`

> ⚠️ **Modèle imposé** : `all-MiniLM-L6-v2` — le même que celui utilisé pour pré-calculer les vecteurs stockés en base.  
> Utiliser un autre modèle invaliderait la comparaison cosinus car les espaces vectoriels seraient incompatibles.

In [ ]:
print(f"⏳ Chargement du modèle : {EMBEDDING_MODEL}…")
model = SentenceTransformer(EMBEDDING_MODEL)

# Vérification de la dimension
test_vec = model.encode("test", normalize_embeddings=True)
assert len(test_vec) == EMBEDDING_DIM, f"Dimension inattendue : {len(test_vec)}"

print(f"✅ Modèle chargé avec succès.")
print(f"   Nom       : {EMBEDDING_MODEL}")
print(f"   Dimension : {len(test_vec)}  (attendu : {EMBEDDING_DIM}) ✓")


## 4. Base vectorielle pré-construite (rappel)

Les fiches techniques (enzymes, améliorants, agents oxydants…) ont **déjà** été :

1. Converties en texte  
2. Découpées en fragments (chunks)  
3. Encodées avec `all-MiniLM-L6-v2`  
4. Stockées dans la colonne `vecteur VECTOR(384)` de la table `embeddings`

➡️ Cette étape est déjà faite. La prochaine étape encode uniquement **la question utilisateur**.

In [ ]:
# Vérification de la dimension des vecteurs stockés en base
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            COUNT(*)                       AS nb_fragments,
            COUNT(DISTINCT id_document)    AS nb_documents,
            vector_dims(vecteur)           AS dimension
        FROM embeddings
        LIMIT 1;
    """)
    stats = cur.fetchone()

print("📊 État du Vector Store en base :")
print(f"   Fragments     : {stats[0]:,}")
print(f"   Documents     : {stats[1]:,}")
print(f"   Dimension     : {stats[2]}  (attendu : {EMBEDDING_DIM}) ✓")


## 5. Saisie de la question utilisateur & génération de l'embedding

> ✏️ Modifiez la variable `USER_QUERY` ci-dessous pour tester différentes questions.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ✏️  EXEMPLE DE QUESTION (modifiable)
USER_QUERY = (
    "Améliorant de panification : quelles sont les quantités recommandées "
    "d'alpha-amylase, xylanase et d'Acide ascorbique ?"
)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"❓ Question  : {USER_QUERY}\n")

# Génération de l'embedding de la question (L2-normalisé)
query_vector = model.encode(USER_QUERY, normalize_embeddings=True)
query_vector_list = query_vector.tolist()   # format attendu par pgvector

print(f"✅ Embedding généré.")
print(f"   Dimension : {len(query_vector)}  |  Norme L2 : {float(np.linalg.norm(query_vector)):.6f}  (≈ 1.0)")


## 6. Calcul de la similarité cosinus via pgvector

pgvector expose l'opérateur `<=>` qui calcule la **distance cosinus** :

$$\text{distance}(Q, D_i) = 1 - \frac{Q \cdot D_i}{\|Q\| \cdot \|D_i\|}$$

On veut la **similarité** (plus élevée = plus pertinent), donc :

$$\text{score} = 1 - \text{distance} = \frac{Q \cdot D_i}{\|Q\| \cdot \|D_i\|}$$

La requête SQL classe directement par `score DESC` et retourne les `TOP_K = 3` premiers fragments.

In [ ]:
SQL = """
    SELECT
        id,
        id_document,
        texte_fragment,
        1 - (vecteur <=> %s::vector)   AS score
    FROM embeddings
    ORDER BY score DESC
    LIMIT %s;
"""

with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute(SQL, (query_vector_list, TOP_K))
    rows = cur.fetchall()

# Construction d'un DataFrame pour inspection
df_results = pd.DataFrame([
    {
        "rank":           i + 1,
        "id":             row["id"],
        "id_document":    row["id_document"],
        "score":          round(float(row["score"]), 4),
        "texte_apercu":   row["texte_fragment"][:80],
    }
    for i, row in enumerate(rows)
])

print(f"✅ Requête exécutée — {len(rows)} résultat(s) retourné(s).\n")
print(df_results.to_string(index=False))


## 7. Affichage des 3 fragments les plus pertinents

Format attendu :

```
Résultat 1
Texte  : "Dosage recommandé : 0.005% à 0.02% du poids de farine."
Score  : 0.91

Résultat 2
Texte  : "Alpha-amylase : utilisation entre 5 et 20 ppm selon la farine."
Score  : 0.87

Résultat 3
Texte  : "Xylanase : améliore l'extensibilité de la pâte..."
Score  : 0.82
```

In [ ]:
print("═" * 60)
print(f"QUESTION : {USER_QUERY}")
print("═" * 60)

for i, row in enumerate(rows, start=1):
    score = float(row["score"])
    texte = row["texte_fragment"].strip()
    print(f"\nRésultat {i}")
    print(f"Texte  : \"{texte}\"")
    print(f"Score  : {score:.2f}")

print()


## 8. Fonction réutilisable — `semantic_search_pg()`

Encapsule le pipeline complet : encode la question → requête SQL → retourne les résultats triés.

In [ ]:
def semantic_search_pg(
    question: str,
    conn,
    model: SentenceTransformer,
    top_k: int = 3,
) -> list[dict]:
    """
    Pipeline RAG complet sur PostgreSQL + pgvector.

    Étapes :
      1. Encode la question avec all-MiniLM-L6-v2
      2. Calcule la similarité cosinus via SQL (opérateur <=>)
      3. Retourne les top_k fragments classés par score décroissant

    Args:
        question : Question en langage naturel.
        conn     : Connexion psycopg2 active.
        model    : SentenceTransformer all-MiniLM-L6-v2.
        top_k    : Nombre de résultats (défaut : 3).

    Returns:
        Liste de dicts : [{"rank", "id", "id_document", "texte_fragment", "score"}, …]
    """
    # 1 — Embedding de la question
    q_vec = model.encode(question, normalize_embeddings=True).tolist()

    # 2 — Requête SQL cosine similarity
    sql = """
        SELECT id, id_document, texte_fragment,
               1 - (vecteur <=> %s::vector) AS score
        FROM embeddings
        ORDER BY score DESC
        LIMIT %s;
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, (q_vec, top_k))
        rows = cur.fetchall()

    # 3 — Résultats structurés
    return [
        {
            "rank":           i + 1,
            "id":             row["id"],
            "id_document":    row["id_document"],
            "texte_fragment": row["texte_fragment"],
            "score":          round(float(row["score"]), 4),
        }
        for i, row in enumerate(rows)
    ]


def display_results(question: str, results: list[dict]) -> None:
    """Affiche les résultats au format demandé."""
    print("═" * 60)
    print(f"QUESTION : {question}")
    print("═" * 60)
    for r in results:
        print(f"\nRésultat {r['rank']}")
        print(f"Texte  : \"{r['texte_fragment'].strip()}\"")
        print(f"Score  : {r['score']:.2f}")
    print()


# ── Exemples de tests ────────────────────────────────────────────────────────
test_questions = [
    "Améliorant de panification : quelles sont les quantités recommandées d'alpha-amylase, xylanase et d'Acide ascorbique ?",
    "Quel est le rôle de la xylanase dans la pâte à pain ?",
    "Dosage de l'acide ascorbique pour un pain de mie ?",
]

for q in test_questions:
    results = semantic_search_pg(q, conn, model, top_k=TOP_K)
    display_results(q, results)


## 9. Fermeture de la connexion & récapitulatif

In [ ]:
conn.close()
print("🔐 Connexion PostgreSQL fermée.")
print()
print("✅ Pipeline RAG complet :")
print(f"   1. Modèle      : {EMBEDDING_MODEL}  (dim={EMBEDDING_DIM})")
print( "   2. Stockage    : PostgreSQL — table embeddings VECTOR(384)")
print( "   3. Similarité  : Cosine Similarity via pgvector (<=>)")
print(f"   4. Résultats   : Top K = {TOP_K}")
